# Calculate Log2 Fold-Change on sum of lipid species
Normalize dataset

In [1]:
import pandas as pd
import numpy as np

In [25]:
# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/raw_data/somme_des_espèces_lipides.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données des onglets
sum_species = pd.read_excel(xls, sheet_name="sum_species")
patient_data = pd.read_excel(xls, sheet_name="patient_data")

In [ ]:
sum_specie

,Family,BS-064D00,BS-064D03,BS-064D10,BS-064D60,BS-082D00,BS-082D03,BS-082D10,BS-082D60,BS-336D00,...,KT-880D10,KT-880D60,KT-926D00,KT-926D03,KT-926D10,KT-926D60,KT-974D00,KT-974D03,KT-974D10,KT-974D60
0,DG,9.534447,1.996816e+01,1.714387e+01,1.277515e+01,7.624890e+00,3.117386e+00,8.932488e+00,8.009256,4.722339e+00,...,2.791721e+01,1.909307e+01,2.957316e+01,1.244153e+01,1.263445e+01,7.164444,2.656210e+00,10.414962,7.077127e+00,7.060300e+00
1,CAR,0.179159,5.535055e-10,7.689679e-10,2.987757e-01,6.184447e-01,1.340882e-09,1.579891e-01,0.909231,8.838220e-01,...,3.337684e-01,1.398908e-01,4.399715e-01,6.444222e-10,7.125891e-10,0.132044,1.027213e+00,0.817418,7.230077e-10,5.259467e-10
2,CE,7.760417,2.848492e+00,1.591850e-09,1.684622e+00,2.880691e+00,9.033381e-01,3.498020e+00,5.314984,2.032174e+00,...,2.281716e-09,2.200704e-09,1.295225e-09,4.419864e+00,1.516224e-09,6.309499,4.970909e+00,8.515708,1.723742e-09,1.383789e+00
3,Cer,5.105788,4.020806e+00,2.693990e+00,2.163743e+00,2.087390e+00,1.348513e+00,1.487014e+00,2.408350,1.360267e+00,...,1.794931e+00,7.637795e-01,1.504139e+00,5.639939e+00,2.856815e+00,1.912225,1.356200e+00,2.048494,2.011308e+00,1.030938e+00
4,Chol,11.286518,8.622505e+00,7.239538e+00,1.316113e+01,6.291542e+00,3.158353e+00,3.166453e+00,7.286367,6.413234e+00,...,8.229098e+00,7.472027e+00,3.479196e+00,1.055920e+01,4.371859e+00,5.183530,2.290283e+00,4.831007,5.124208e+00,2.352664e-10
5,HexCer,4.688535,1.200005e+00,2.637131e-09,1.978631e-09,1.523926e-09,3.434066e-09,1.952362e-09,2.840746,2.238138e-09,...,2.455796e-09,2.535497e-09,1.318913e-09,2.483256e+00,2.446184e-09,1.354952,1.652893e-09,2.442280,2.380952e-09,1.671123e-09
6,LPC,1595.862814,3.055236e+03,5.249176e+03,4.157752e+03,1.178461e+03,1.382943e+03,2.382797e+03,4629.368659,2.484710e+03,...,2.537056e+03,2.453207e+03,2.560285e+03,1.951959e+03,3.808740e+03,2919.047526,8.564942e+02,2498.226063,3.253496e+03,3.843256e+03
7,PC,1230.068372,1.291379e+03,1.853937e+03,1.001896e+03,5.690893e+02,4.873015e+02,6.964489e+02,1397.529059,9.946018e+02,...,2.104073e+03,1.883833e+03,9.536159e+02,9.036817e+02,1.042136e+03,1023.194735,3.840497e+02,668.418430,1.115958e+03,1.439130e+03
8,PE,7.655925,6.649396e+00,9.021984e+00,2.046131e-08,1.157895e-08,4.579517e-08,1.824414e-08,15.256146,1.691094e-08,...,1.483109e+01,1.275616e+01,9.166667e-09,1.691094e-08,2.249795e-08,20.829557,1.122449e-08,4.917157,2.013177e-08,2.799457e-08
9,SM,470.149453,2.465139e+02,2.424261e+02,2.296680e+02,1.973391e+02,1.406332e+02,1.075548e+02,264.277949,2.626944e+02,...,3.522950e+02,3.274698e+02,1.431475e+02,3.486843e+02,1.920323e+02,238.563650,1.604170e+02,217.820758,1.818897e+02,1.826081e+02


In [27]:
sum_species.dtypes

Family        object
BS-064D00    float64
BS-064D03    float64
BS-064D10    float64
BS-064D60    float64
              ...   
KT-926D60    float64
KT-974D00    float64
KT-974D03    float64
KT-974D10    float64
KT-974D60    float64
Length: 161, dtype: object

In [28]:
# Extraire les noms de colonnes correspondant aux échantillons
sample_columns = [
    col for col in sum_species.columns if "D0" in col or "D03" in col or "D10" in col or "D60" in col]

In [29]:
# Créer de nouvelles colonnes avec le log2 fold-change
for col in sample_columns:
    patient_id = col[:-3]  # Retirer le suffixe temporel (D0, D03, etc.)
    if f"{patient_id}D60" in sum_species.columns:  # Vérifier que D60 existe pour ce patient
        sum_species[f"log2FC_D00-{patient_id}"] = np.log2(
            (sum_species[f"{patient_id}D00"]) / (sum_species[f"{patient_id}D60"]))
        sum_species[f"log2FC_D03-{patient_id}"] = np.log2(
            (sum_species[f"{patient_id}D03"]) / (sum_species[f"{patient_id}D60"]))
        sum_species[f"log2FC_D10-{patient_id}"] = np.log2(
            (sum_species[f"{patient_id}D10"]) / (sum_species[f"{patient_id}D60"]))

/var/folders/d3/l5snwfsj7yb7zrln2fvzwpqr0000gn/T/ipykernel_90515/1671599876.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sum_species[f"log2FC_D10-{patient_id}"] = np.log2(
/var/folders/d3/l5snwfsj7yb7zrln2fvzwpqr0000gn/T/ipykernel_90515/1671599876.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sum_species[f"log2FC_D00-{patient_id}"] = np.log2(
/var/folders/d3/l5snwfsj7yb7zrln2fvzwpqr0000gn/T/ipykernel_90515/1671599876.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fr

In [38]:
log2fc_cols = [
    col for col in sum_species.columns if "log2FC" in col]
family_cols = [
    col for col in sum_species.columns if "Family" in col]
log2fc_data = sum_species[family_cols + log2fc_cols]
log2fc_data

,Family,log2FC_D00-BS-064,log2FC_D03-BS-064,log2FC_D10-BS-064,log2FC_D00-BS-082,log2FC_D03-BS-082,log2FC_D10-BS-082,log2FC_D00-BS-336,log2FC_D03-BS-336,log2FC_D10-BS-336,...,log2FC_D10-KT-805,log2FC_D00-KT-880,log2FC_D03-KT-880,log2FC_D10-KT-880,log2FC_D00-KT-926,log2FC_D03-KT-926,log2FC_D10-KT-926,log2FC_D00-KT-974,log2FC_D03-KT-974,log2FC_D10-KT-974
0,DG,-0.422119,0.644361,0.424353,-0.070952,-1.361331,0.157394,-1.531951,0.247795,0.300514,...,1.405524,-0.821127,-1.170414,0.548105,2.045362,0.796237,0.818436,-1.410360,0.560856,0.003434
1,CAR,-0.737824,-29.007818,-28.533493,-0.556003,-29.336890,-2.524823,0.457988,0.074914,-2.280652,...,-1.849496,-27.305830,-27.493873,1.254546,1.736387,-27.610370,-27.465305,30.863099,30.533511,0.459094
2,CE,2.203710,0.757774,-29.979073,-0.883650,-2.556727,-0.603527,-0.091410,1.209698,1.391625,...,0.454442,-0.310879,27.739847,0.052154,-32.181676,-0.513524,-31.954396,1.844886,2.621503,-29.580433
3,Cer,1.238605,0.893956,0.316215,-0.206345,-0.836675,-0.695627,0.658680,1.057277,1.615020,...,0.071424,1.179902,0.613199,1.232700,-0.346314,1.560427,0.579156,0.395613,0.990607,0.964177
4,Chol,-0.221683,-0.610104,-0.862314,-0.211786,-1.206027,-1.202332,-1.135507,-0.899032,-1.106105,...,-0.369241,-0.183247,-1.436146,0.139235,-0.575181,1.026493,-0.245688,33.180511,34.257309,34.342315
5,HexCer,31.141988,29.175891,0.414467,-30.795830,-29.623705,-30.438402,0.259910,28.987915,29.235671,...,-29.196029,0.197251,-0.375357,-0.046078,-29.936244,0.873992,-29.045061,-0.015825,30.444764,0.510721
6,LPC,-1.381467,-0.444520,0.336287,-1.973912,-1.743074,-0.958160,-0.904368,-0.673251,-0.050098,...,0.429789,-0.932435,-2.910540,0.048487,-0.189193,-0.580575,0.383816,-2.165814,-0.621425,-0.240338
7,PC,0.296005,0.366179,0.887858,-1.296151,-1.519992,-1.004789,-0.271917,-0.318199,0.317648,...,0.034722,-0.995652,-1.282090,0.159514,-0.101600,-0.179194,0.026463,-1.905832,-1.106374,-0.366915
8,PE,28.479103,28.275749,28.715971,-30.295239,-28.311548,-29.639310,0.274277,0.328054,0.311270,...,-29.407692,-0.585332,-29.548981,0.217431,-31.081516,-30.198028,-29.786191,-1.318497,27.388102,-0.475673
9,SM,1.033569,0.102119,0.077995,-0.421379,-0.910119,-1.296984,-0.007940,-0.181974,-0.210063,...,-0.187187,-0.347450,-1.355368,0.105422,-0.736872,0.547547,-0.313026,-0.186924,0.254391,-0.005686


In [39]:
# Sauvegarder le nouveau fichier
output_file = "/Users/loictalignani/research/project/lipidomics/data/pretreatment/log2FC_somme_des_especes_lipides.tsv"
log2fc_data.to_csv(output_file, sep="\t", index=False)

print(f"Fichier enregistré : {output_file}")

Fichier enregistré : /Users/loictalignani/research/project/lipidomics/data/pretreatment/log2FC_somme_des_especes_lipides.tsv


In [40]:
log2fc_cols_D0 = [
    col for col in log2fc_data.columns if "D00" in col]
log2fc_cols_D03 = [
    col for col in log2fc_data.columns if "D03" in col]
log2fc_cols_D10 = [
    col for col in log2fc_data.columns if "D10" in col]

In [41]:
log2fc_data_D0 = log2fc_data[family_cols + log2fc_cols_D0]
log2fc_data_D03 = log2fc_data[family_cols + log2fc_cols_D03]
log2fc_data_D10 = log2fc_data[family_cols + log2fc_cols_D10]

In [42]:
# Sauvegarder le nouveau fichier
outfileD0 = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/log2FC_somme_des_especes_lipide_D0.tsv"
log2fc_data_D0.to_csv(outfileD0, sep="\t", index=False)

print(f"Fichier enregistré : {outfileD0}")

# Sauvegarder le nouveau fichier
outfileD03 = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/log2FC_somme_des_especes_lipide_D03.tsv"
log2fc_data_D03.to_csv(outfileD03, sep="\t", index=False)

print(f"Fichier enregistré : {outfileD03}")

# Sauvegarder le nouveau fichier
outfileD10 = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/log2FC_somme_des_especes_lipide_D10.tsv"
log2fc_data_D10.to_csv(outfileD10, sep="\t", index=False)

print(f"Fichier enregistré : {outfileD10}")

Fichier enregistré : /Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/log2FC_somme_des_especes_lipide_D0.tsv
Fichier enregistré : /Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/log2FC_somme_des_especes_lipide_D03.tsv
Fichier enregistré : /Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/log2FC_somme_des_especes_lipide_D10.tsv
